# 🤖 Algorithmes de Machine Learning en Python
## Module Data Science

> **Prérequis :** NumPy, Pandas, Visualisation

---

Notebook **interactif**. Exécutez chaque cellule avec `Shift + Enter`.
Certains exemples utilisent `bootcamp_500.csv` (à télécharger dans Colab).

### Table des matières
1. Introduction au ML
2. Régression linéaire
3. Arbres de décision
4. KNN
5. Comparaison
6. Exercices


In [ ]:
!pip install scikit-learn seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
print("✅ Bibliothèques prêtes")

---
## 1. Introduction au Machine Learning

Le ML **apprend des schémas** à partir de données au lieu d'être programmé avec des règles.

- **Supervisé** : régression (prédire un nombre) vs classification (prédire une catégorie)
- **Workflow** : collecter → nettoyer → split → fit → predict → évaluer
- **Interface sklearn** : `model.fit(X, y)` puis `model.predict(X)`

In [ ]:
# Le train_test_split : séparer données d'apprentissage et de test
from sklearn.model_selection import train_test_split

# Exemple minimal
X_demo = np.arange(20).reshape(-1, 1)
y_demo = np.arange(20)
X_tr, X_te, y_tr, y_te = train_test_split(X_demo, y_demo, test_size=0.2, random_state=42)
print(f"Train : {len(X_tr)} | Test : {len(X_te)}")

---
## 2. Régression linéaire

Prédit un NOMBRE via la droite de meilleur ajustement (moindres carrés).
Équation : **y = β₀ + β₁·x**

In [ ]:
# Données : heures d'étude → notes
np.random.seed(42)
heures = np.random.rand(100, 1) * 12
notes = 1.3 * heures.flatten() + 3 + np.random.randn(100) * 1.5

X_train, X_test, y_train, y_test = train_test_split(heures, notes, test_size=0.2, random_state=42)

modele = LinearRegression()
modele.fit(X_train, y_train)

print(f"Pente (β₁)    : {modele.coef_[0]:.2f}")
print(f"Ordonnée (β₀) : {modele.intercept_:.2f}")
print(f"Prédiction 10h : {modele.predict([[10]])[0]:.1f}/20")

In [ ]:
# Visualiser la droite de meilleur ajustement
plt.figure(figsize=(8, 5))
plt.scatter(heures, notes, alpha=0.5, label="Données réelles")
x_line = np.linspace(0, 12, 100).reshape(-1, 1)
plt.plot(x_line, modele.predict(x_line), color="red", linewidth=2, label="Droite de régression")
plt.xlabel("Heures d'étude"); plt.ylabel("Note")
plt.title("Régression linéaire : heures d'étude → note")
plt.legend()
plt.show()

In [ ]:
# Évaluer le modèle
predictions = modele.predict(X_test)
print(f"R²   : {r2_score(y_test, predictions):.3f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, predictions)):.2f}")

### Régression multiple (plusieurs features)

In [ ]:
# Utiliser bootcamp_500.csv (télécharger dans Colab)
df = pd.read_csv("bootcamp_500.csv")
df.loc[df["note_sql"] > 20, "note_sql"] = np.nan
df = df.dropna(subset=["note_sql", "note_python", "heures_etude", "age"])

X = df[["heures_etude", "age", "note_python"]]
y = df["note_sql"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modele = LinearRegression().fit(X_train, y_train)
for nom, coef in zip(X.columns, modele.coef_):
    print(f"{nom:15} : {coef:.3f}")
print(f"R² : {modele.score(X_test, y_test):.3f}")

---
## 3. Arbres de décision

Série de questions oui/non. Interprétable et visualisable. Attention à l'overfitting (élagage).

In [ ]:
df3 = df.dropna(subset=["note_sql", "note_python", "heures_etude"])
X = df3[["heures_etude", "note_sql", "note_python"]]
y = df3["niveau"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

arbre = DecisionTreeClassifier(max_depth=3, random_state=42)
arbre.fit(X_train, y_train)
print(f"Précision : {accuracy_score(y_test, arbre.predict(X_test)):.3f}")

In [ ]:
# Visualiser l'arbre (grand atout !)
plt.figure(figsize=(16, 8))
plot_tree(arbre, filled=True, feature_names=X.columns,
          class_names=sorted(y.unique()), rounded=True, fontsize=9)
plt.title("Arbre de décision")
plt.show()

In [ ]:
# Effet de l'élagage : tester plusieurs profondeurs
for prof in [1, 2, 3, 5, 10, None]:
    a = DecisionTreeClassifier(max_depth=prof, random_state=42).fit(X_train, y_train)
    acc_train = accuracy_score(y_train, a.predict(X_train))
    acc_test  = accuracy_score(y_test, a.predict(X_test))
    print(f"max_depth={str(prof):5} | train={acc_train:.3f} | test={acc_test:.3f}")
# → observez l'overfitting quand la profondeur augmente trop

---
## 4. KNN (K plus proches voisins)

Vote des K voisins les plus proches. **Normalisation OBLIGATOIRE**.

In [ ]:
df4 = df.dropna(subset=["note_sql", "note_python", "heures_etude", "salaire_stage"])
X = df4[["heures_etude", "note_sql", "note_python", "salaire_stage"]]
y = df4["niveau"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# SANS normalisation
knn_brut = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print(f"Sans normalisation : {accuracy_score(y_test, knn_brut.predict(X_test)):.3f}")

# AVEC normalisation
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
knn_norm = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
print(f"Avec normalisation : {accuracy_score(y_test, knn_norm.predict(X_test_s)):.3f}")

In [ ]:
# Trouver le meilleur K
scores = {}
for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    scores[k] = accuracy_score(y_test, knn.predict(X_test_s))

plt.plot(list(scores.keys()), list(scores.values()), marker="o")
plt.xlabel("K"); plt.ylabel("Précision"); plt.title("Précision selon K")
plt.grid(alpha=0.3); plt.show()

meilleur_k = max(scores, key=scores.get)
print(f"Meilleur K : {meilleur_k} ({scores[meilleur_k]:.3f})")

---
## 5. Comparaison des trois algorithmes

| | Régression linéaire | Arbre | KNN |
|-|-----|-----|-----|
| Prédit | nombre | catégorie/nombre | catégorie/nombre |
| Normalisation | recommandée | non | **obligatoire** |
| Interprétable | ✅✅✅ | ✅✅✅ | ❌ |
| Non-linéaire | ❌ | ✅ | ✅ |

→ Il n'y a pas de meilleur algo universel : on teste et on compare !

---
## 6. 💻 Exercices

Complétez, puis comparez avec la solution.

### Exercice 1 — Régression simple (note_sql ~ heures_etude)

In [ ]:
# ✏️ À vous de jouer !



In [ ]:
# ✅ Solution
df = pd.read_csv("bootcamp_500.csv")
df.loc[df["note_sql"]>20,"note_sql"]=np.nan
df = df.dropna(subset=["note_sql","heures_etude"])
X, y = df[["heures_etude"]], df["note_sql"]
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42)
m = LinearRegression().fit(Xtr,ytr)
print(f"Pente {m.coef_[0]:.3f} | Ordonnée {m.intercept_:.3f} | R² {m.score(Xte,yte):.3f}")

### Exercice 2 — Arbre pour prédire le niveau

In [ ]:
# ✏️ À vous de jouer !



In [ ]:
# ✅ Solution
df3 = df.dropna(subset=["note_sql","note_python","heures_etude"])
X, y = df3[["heures_etude","note_sql","note_python"]], df3["niveau"]
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42)
a = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr,ytr)
print(f"Précision : {accuracy_score(yte, a.predict(Xte)):.3f}")

### Exercice 3 — KNN normalisé + meilleur K

In [ ]:
# ✏️ À vous de jouer !



In [ ]:
# ✅ Solution
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr); Xte_s = scaler.transform(Xte)
sc = {}
for k in range(1,21):
    knn = KNeighborsClassifier(n_neighbors=k).fit(Xtr_s, ytr)
    sc[k] = accuracy_score(yte, knn.predict(Xte_s))
bk = max(sc, key=sc.get)
print(f"Meilleur K : {bk} ({sc[bk]:.3f})")

---
## 🎉 Félicitations !

Vous savez implémenter les 3 algorithmes fondamentaux du ML.
**Prochaine étape :** Checkpoint 5 — le dataset Iris !

*📘 Module Data Science — Algorithmes de ML | Bootcamp Data Science*